In [7]:
# 01 — Xác định root dự án trên Windows / VS Code (không cần Colab, không tải gì từ mạng)
import os, sys, gc, json
from pathlib import Path
import torch, numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

def locate_project_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p/'src'/'models'/'builder.py').is_file() and (p/'data'/'processed').is_dir():
            return p
    raise FileNotFoundError(
        f'Không tìm thấy root Chest-Xray-Imbalance từ {here}. '
        'Hãy mở VS Code tại thư mục gốc dự án rồi chạy notebook.'
    )

PROJECT_ROOT=locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Project:', PROJECT_ROOT)
print('Device:', DEVICE)
if DEVICE.type=='cuda':
    print('GPU:',torch.cuda.get_device_name(0))
    total=torch.cuda.get_device_properties(0).total_memory/1024**3
    print(f'VRAM: {total:.2f} GiB')
else:
    print('CUDA không có; vẫn có thể chạy trên CPU nhưng chậm.')


Project: D:\Theory\Chest-Xray-Imbalance
Device: cuda:0
GPU: NVIDIA GeForce GTX 1060 3GB
VRAM: 3.00 GiB


In [8]:
# 02 — Cấu hình local (đủ 5 seed trên cả 2 dataset)
SEEDS = [7, 42, 132, 2024, 3407]
IMAGE_SIZE = 512
BATCH_SIZE = 1           # RTX 3050 4GB; chỉ 1 DenseNet201 đang nằm trên GPU
N_CASES_PER_DATASET = 2   # Đánh giá tất cả ảnh dương để chọn; Grad-CAM trên 2 ca mỗi bộ
SAVE_INDIVIDUAL_SEED_FIGURES = True

DATASETS = {
    'VinDr-CXR': {
        'label': 'Pneumothorax',
        'config_names': ['bce', 'wbce_w1', 'focal_f2', 'asl_a2'],
        'model_titles': ['BCE', 'WBCE W1', 'Focal F2', 'ASL A2'],
    },
    'VinDr-PCXR': {
        'label': 'Hyaline membrane disease',
        'config_names': ['bce', 'wbce_w1', 'focal_f1', 'asl_a2'],
        'model_titles': ['BCE', 'WBCE W1', 'Focal F1', 'ASL A2'],
    },
}
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed'
CKPT_ROOT = PROJECT_ROOT / 'outputs' / 'checkpoints' / 'stage1-Loss_Functions'
CSV_ROOTS = [PROJECT_ROOT / 'outputs' / 'csv' / 'stage1-Loss_Functions']
OUT_ROOT = PROJECT_ROOT / 'outputs' / 'visualizations' / 'stage1-Loss_Functions' / 'Grad_CAM_5seeds'
OUT_ROOT.mkdir(parents=True,exist_ok=True)
assert SEEDS == [7, 42, 132, 2024, 3407]
assert CKPT_ROOT.is_dir(), f'Stage 1 checkpoint root missing: {CKPT_ROOT}'
assert CSV_ROOTS[0].is_dir(), f'Stage 1 CSV directory missing: {CSV_ROOTS[0]}'
print('Data:',DATA_ROOT)
print('Checkpoints:',CKPT_ROOT)
print('CSV search paths:',CSV_ROOTS)
print('Figures + reports:',OUT_ROOT)

assert 'stage1-Loss_Functions' in CKPT_ROOT.parts and 'stage2-Representation' not in CKPT_ROOT.parts


Data: D:\Theory\Chest-Xray-Imbalance\data\processed
Checkpoints: D:\Theory\Chest-Xray-Imbalance\outputs\checkpoints\stage1-Loss_Functions
CSV search paths: [WindowsPath('D:/Theory/Chest-Xray-Imbalance/outputs/csv/stage1-Loss_Functions')]
Figures + reports: D:\Theory\Chest-Xray-Imbalance\outputs\visualizations\stage1-Loss_Functions\Grad_CAM_5seeds


In [9]:
# 03 — Import đúng DenseNet201 Stage 1 và xác minh lớp Grad-CAM
from torchvision import models
from src.data.dataset import CLASSES
from src.data.transforms import get_transforms, IMAGENET_MEAN, IMAGENET_STD
from src.models.builder import build_model

for dataset,spec in DATASETS.items():
    assert len(CLASSES[dataset])==14 and spec['label'] in CLASSES[dataset]

probe=build_model('densenet201',num_classes=14,pretrained=False)
assert hasattr(probe.features,'denseblock4')
del probe
print('[PASS] Đúng kiến trúc DenseNet201 Stage 1, 14 nhãn, target = features.denseblock4')


[PASS] Đúng kiến trúc DenseNet201 Stage 1, 14 nhãn, target = features.denseblock4


In [10]:
# 04 — Đọc Test trực tiếp từ thư mục local; kiểm tra số lượng và trùng Train / Val / Test
EXPECTED_TEST={'VinDr-CXR':3000,'VinDr-PCXR':1397}
IMAGE_EXTS={'.png','.jpg','.jpeg','.bmp'}

def get_root(dataset):
    base=DATA_ROOT/dataset
    candidates=[base, base/dataset]
    ok=[p for p in candidates if (p/'test').is_dir() and (p/'labels'/'test_labels.csv').is_file()]
    if len(ok)!=1:
        raise FileNotFoundError(f'{dataset}: cần duy nhất một root chứa labels/test_labels.csv + test/. Đã tìm: {candidates}')
    return ok[0]

ROOTS={dataset:get_root(dataset) for dataset in DATASETS}
SAMPLES={}
for dataset,root in ROOTS.items():
    df=pd.read_csv(root/'labels'/'test_labels.csv',dtype={'image_id':str})
    cls=CLASSES[dataset]
    assert len(df)==EXPECTED_TEST[dataset], (dataset,'Sai số dòng Test',len(df))
    assert df.image_id.notna().all() and df.image_id.is_unique
    assert set(cls).issubset(df.columns)
    assert np.isin(df[cls].to_numpy(dtype=np.float32),[0,1]).all()
    train_ids=set(pd.read_csv(root/'labels'/'train_split.csv',dtype={'image_id':str}).image_id)
    val_ids=set(pd.read_csv(root/'labels'/'val_split.csv',dtype={'image_id':str}).image_id)
    assert set(df.image_id).isdisjoint(train_ids|val_ids), f'{dataset}: Trùng Test với Train/Val'
    target=df.loc[df[DATASETS[dataset]['label']].eq(1),'image_id'].tolist()
    assert len(target)>0, f'{dataset}: Không có ảnh Test dương tính cho bệnh đã chọn'
    index={}
    for p in (root/'test').iterdir():
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            assert p.stem not in index, f'Trùng tên ảnh: {p.stem}'
            index[p.stem]=p
    missing=[x for x in target if x not in index]
    assert not missing, f'{dataset}: Thiếu {len(missing)} ảnh, ví dụ {missing[:3]}'
    SAMPLES[dataset]=[(id,index[id]) for id in target]
    print(f'[TEST PASS] {dataset}: {len(df)} ảnh, {len(target)} ảnh dương của {DATASETS[dataset]["label"]}')


[TEST PASS] VinDr-CXR: 3000 ảnh, 18 ảnh dương của Pneumothorax
[TEST PASS] VinDr-PCXR: 1397 ảnh, 3 ảnh dương của Hyaline membrane disease


In [11]:
# 05 — Kiểm toán tất cả checkpoint và CSV trước khi chạy bất cứ Grad-CAM nào
from functools import lru_cache

# Cache danh sách CSV theo đường dẫn thay vì duyệt rglob 40 lần.
CSV_FILES = []
for root in CSV_ROOTS:
    if root.exists():
        CSV_FILES.extend(p.resolve() for p in root.rglob('*per_class.csv') if p.is_file())
        CSV_FILES.extend(p.resolve() for p in root.rglob('per_class.csv') if p.is_file())
CSV_FILES = sorted(set(CSV_FILES))
print('Found Stage1 per-class CSVs:',len(CSV_FILES))

@lru_cache(None)
def read_metrics_csv(path):
    df = pd.read_csv(path)
    required = {'dataset','seed','config','split','label','threshold','AP'}
    return df if required.issubset(df.columns) else None

def get_checkpoint(dataset, config, seed):
    file = f'best_{config}_seed{seed}.pth'
    candidates = [CKPT_ROOT/dataset/f'seed{seed}'/file]
    found = list(dict.fromkeys(p.resolve() for p in candidates if p.is_file()))
    if len(found) != 1:
        raise FileNotFoundError(f'{dataset} seed{seed} {config}: expected exactly 1 checkpoint {file}. Checked: {candidates}')
    return found[0]

def get_perclass_row(dataset, config, seed, label):
    matches = []
    for path in CSV_FILES:
        df = read_metrics_csv(str(path))
        if df is None:
            continue
        sub=df.loc[(df.dataset.astype(str)==dataset) &
                   (pd.to_numeric(df.seed,errors='coerce')==seed) &
                   (df.config.astype(str).str.lower()==config) &
                   (df.split.astype(str).str.lower()=='test') &
                   (df.label.astype(str)==label)]
        if len(sub)==1:
            matches.append((path,sub.iloc[0]))
    if not matches:
        raise FileNotFoundError(f'Missing Stage 1 test per-class CSV: {dataset}, seed {seed}, {config}, {label}. '
                                'Copy Stage 1 per_class.csv from Drive into outputs/csv/stage1-Loss_Functions/.')
    first_path, first_row=matches[0]
    for other_path,other_row in matches[1:]:
        for field in ('threshold','AP'):
            a=pd.to_numeric(pd.Series([first_row[field]]),errors='coerce').iloc[0]
            b=pd.to_numeric(pd.Series([other_row[field]]),errors='coerce').iloc[0]
            assert np.isclose(a,b,equal_nan=True,atol=1e-8), (
                f'Conflicting {field} in {first_path} vs {other_path}: {dataset}/{config}/seed{seed}')
    return first_path,first_row

CHECKPOINTS = {}
THRESHOLDS = {}
AP_SCORES = {}
CSV_SOURCES = {}
missing=[]
for dataset,spec in DATASETS.items():
    for seed in SEEDS:
        for config in spec['config_names']:
            key=(dataset,seed,config)
            try:
                cp=get_checkpoint(dataset,config,seed)
                csv_path,row=get_perclass_row(dataset,config,seed,spec['label'])
                th,ap=float(row.threshold),float(row.AP)
                assert np.isfinite(th) and 0<=th<=1
                assert np.isfinite(ap) and 0<=ap<=1, f'AP undefined for {dataset}/{seed}/{config}'
                CHECKPOINTS[key]=cp; THRESHOLDS[key]=th
                AP_SCORES[key]=ap; CSV_SOURCES[key]=csv_path
            except (FileNotFoundError,ValueError,AssertionError) as ex:
                missing.append(str(ex))
if missing:
    print('\n'.join('[MISSING] '+m for m in missing))
    raise RuntimeError(f'Missing or inconsistent resources: {len(missing)} items. Fix before inference.')
assert len(CHECKPOINTS)==len(DATASETS)*len(SEEDS)*4
print('[PASS] Validated',len(CHECKPOINTS),'checkpoints + frozen thresholds + Test AP values')


Found Stage1 per-class CSVs: 0
[MISSING] Missing Stage 1 test per-class CSV: VinDr-CXR, seed 7, bce, Pneumothorax. Copy Stage 1 per_class.csv from Drive into outputs/csv/stage1-Loss_Functions/.
[MISSING] Missing Stage 1 test per-class CSV: VinDr-CXR, seed 7, wbce_w1, Pneumothorax. Copy Stage 1 per_class.csv from Drive into outputs/csv/stage1-Loss_Functions/.
[MISSING] Missing Stage 1 test per-class CSV: VinDr-CXR, seed 7, focal_f2, Pneumothorax. Copy Stage 1 per_class.csv from Drive into outputs/csv/stage1-Loss_Functions/.
[MISSING] Missing Stage 1 test per-class CSV: VinDr-CXR, seed 7, asl_a2, Pneumothorax. Copy Stage 1 per_class.csv from Drive into outputs/csv/stage1-Loss_Functions/.
[MISSING] Missing Stage 1 test per-class CSV: VinDr-CXR, seed 42, bce, Pneumothorax. Copy Stage 1 per_class.csv from Drive into outputs/csv/stage1-Loss_Functions/.
[MISSING] Missing Stage 1 test per-class CSV: VinDr-CXR, seed 42, wbce_w1, Pneumothorax. Copy Stage 1 per_class.csv from Drive into outputs/c

RuntimeError: Missing or inconsistent resources: 40 items. Fix before inference.

In [ ]:
# 06 — Load mô hình Stage 1 tuần tự; cùng preprocess với training
import gc

def open_model(dataset, config, seed):
    path=CHECKPOINTS[(dataset,seed,config)]
    payload=torch.load(path,map_location='cpu',weights_only=True)
    if isinstance(payload,dict) and payload and all(torch.is_tensor(v) for v in payload.values()):
        state=payload
    elif isinstance(payload,dict) and isinstance(payload.get('model_state_dict'),dict):
        state=payload['model_state_dict']
    else:
        raise ValueError(f'Unexpected Stage 1 checkpoint format: {path}')
    if state and all(k.startswith('module.') for k in state):
        state={k.removeprefix('module.'):v for k,v in state.items()}
    model=build_model('densenet201',num_classes=14,pretrained=False)
    model.load_state_dict(state,strict=True)
    model.to(DEVICE).eval()
    return model

def release_model(model):
    model.cpu()  # free GPU buffers even while caller still references this model
    del model
    gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()

TRANSFORM=get_transforms('val',IMAGE_SIZE)
MEAN=np.asarray(IMAGENET_MEAN,dtype=np.float32).reshape(1,1,3)
STD=np.asarray(IMAGENET_STD,dtype=np.float32).reshape(1,1,3)

def image_tensor(path):
    img=cv2.imread(str(path),cv2.IMREAD_UNCHANGED)
    if img is None:raise RuntimeError(f'Cannot read: {path}')
    if img.ndim==2:img=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
    elif img.ndim==3 and img.shape[2]==3:img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    elif img.ndim==3 and img.shape[2]==4:img=cv2.cvtColor(img,cv2.COLOR_BGRA2RGB)
    else:raise ValueError((str(path),img.shape))
    if img.dtype==np.uint16:img=(img.astype(np.float32)/257).clip(0,255).astype(np.uint8)
    assert img.dtype==np.uint8
    x=TRANSFORM(image=img)['image']
    assert tuple(x.shape)==(3,IMAGE_SIZE,IMAGE_SIZE)
    return x

def display_rgb(x):
    return np.clip(x.cpu().permute(1,2,0).numpy()*STD+MEAN,0,1)


In [ ]:
# 07 — Dự đoán toàn bộ ảnh Test DƯƠNG cho các seed; sau đó chọn ảnh minh họa.
# Lưu ý: đánh giá chỉ trên ảnh dương để CHỌN minh họa, không thể dùng để tính AP Test.
PREDICTIONS={}
PROBABILITY_ROWS=[]
for dataset,spec in DATASETS.items():
    samples=SAMPLES[dataset]
    idx=CLASSES[dataset].index(spec['label'])
    all_ids=[image_id for image_id,_ in samples]
    x_all=torch.stack([image_tensor(path) for _,path in samples],dim=0)
    frame=pd.DataFrame({'image_id':all_ids,'class':spec['label']})
    for seed in SEEDS:
        for config in spec['config_names']:
            model=open_model(dataset,config,seed)
            probs=[]
            with torch.inference_mode():
                for batch in x_all.split(BATCH_SIZE):
                    logits=model(batch.to(DEVICE,non_blocking=True))
                    probs.extend(torch.sigmoid(logits[:,idx]).float().cpu().tolist())
            release_model(model)
            key=(dataset,seed,config)
            th=THRESHOLDS[key]
            frame[f'prob_{config}_seed{seed}']=probs
            frame[f'tp_{config}_seed{seed}']=(np.asarray(probs)>=th).astype(int)
            PROBABILITY_ROWS.extend(dict(dataset=dataset,seed=seed,config=config,image_id=id,
                                         class_name=spec['label'],positive=1,
                                         probability=float(prob),threshold=th,
                                         TP=int(prob>=th)) for id,prob in zip(all_ids,probs))
    for config in spec['config_names']:
        cols=[f'prob_{config}_seed{seed}' for seed in SEEDS]
        tp_cols=[f'tp_{config}_seed{seed}' for seed in SEEDS]
        frame[f'prob_mean_{config}']=frame[cols].mean(axis=1)
        frame[f'prob_std_{config}']=frame[cols].std(axis=1,ddof=1)
        frame[f'tp_seeds_{config}']=frame[tp_cols].sum(axis=1)
    # Rank by BCE failures recovered by alternatives across 5 seeds.
    frame['bce_FN_seeds']=len(SEEDS)-frame['tp_seeds_bce']
    frame['improved_TP_total']=frame[[f'tp_seeds_{x}' for x in spec['config_names'][1:]]].sum(axis=1)
    frame['improvement_vs_bce']=frame['improved_TP_total']-3*frame['tp_seeds_bce']
    frame=frame.sort_values(['improvement_vs_bce','bce_FN_seeds','improved_TP_total','image_id'],
                            ascending=[False,False,False,True]).reset_index(drop=True)
    PREDICTIONS[dataset]=frame
    folder=OUT_ROOT/dataset; folder.mkdir(parents=True,exist_ok=True)
    frame.to_csv(folder/'positive_test_candidates_5seeds.csv',index=False,encoding='utf-8-sig')
    print(f'[POSITIVE CANDIDATES] {dataset}: {len(frame)} Test-positive cases')
    display(frame[['image_id','bce_FN_seeds','improved_TP_total','improvement_vs_bce']+[
        f'tp_seeds_{x}' for x in spec['config_names']]].head(10))

pd.DataFrame(PROBABILITY_ROWS).to_csv(OUT_ROOT/'all_positive_predictions_by_seed.csv',index=False,encoding='utf-8-sig')


In [ ]:
# 08 — Grad-CAM for DenseNet201 (model.features.denseblock4).
# Use autograd.grad() against captured feature tensor; no backward hook/in-place ReLU bug.
import torch.nn.functional as F

class GradCAM:
    def __init__(self,model):
        self.model=model
        self.activation=None
        self.handle=model.features.denseblock4.register_forward_hook(self._capture)
    def _capture(self,module,inp,out):
        self.activation=out
    def run(self,x,class_idx):
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits=self.model(x.unsqueeze(0).to(DEVICE))
            assert logits.shape==(1,14)
            assert self.activation is not None
            grad=torch.autograd.grad(logits[0,class_idx],self.activation,retain_graph=False,create_graph=False)[0]
            weights=grad.mean(dim=(2,3),keepdim=True)
            heat=torch.relu((weights*self.activation).sum(1,keepdim=True))
            heat=F.interpolate(heat,size=(IMAGE_SIZE,IMAGE_SIZE),mode='bilinear',align_corners=False)
            heat=heat[0,0].detach().float().cpu().numpy()
            assert np.isfinite(heat).all(), 'Nonfinite Grad-CAM'
            mn,mx=float(heat.min()),float(heat.max())
            norm=(heat-mn)/(mx-mn+1e-8) if mx>mn else np.zeros_like(heat)
            prob=float(torch.sigmoid(logits[0,class_idx]).detach().item())
        self.activation=None
        return norm,prob
    def close(self):
        self.handle.remove()

from matplotlib import cm

def overlay_cam(rgb,heat,alpha=.47):
    colored=cm.get_cmap('jet')(heat)[...,:3].astype(np.float32)
    return np.clip((1-alpha)*rgb + alpha*colored,0,1)


In [ ]:
# 09 — Xuất Grad-CAM 5 seed/loss, CAM trung bình và báo cáo xác suất/AP mean ± std
import json
from matplotlib import cm

all_selected=[]
all_case_metrics=[]
all_ap=[]
for dataset,spec in DATASETS.items():
    idx=CLASSES[dataset].index(spec['label'])
    folder=OUT_ROOT/dataset;folder.mkdir(parents=True,exist_ok=True)
    for config in spec['config_names']:
        vals=np.array([AP_SCORES[(dataset,seed,config)] for seed in SEEDS],dtype=float)
        all_ap.append(dict(dataset=dataset,label=spec['label'],config=config,
                           n_seeds=len(SEEDS),AP_mean=float(vals.mean()),
                           AP_std=float(vals.std(ddof=1)),AP_min=float(vals.min()),
                           AP_max=float(vals.max())))

    frame=PREDICTIONS[dataset]
    chosen=frame.head(min(N_CASES_PER_DATASET,len(frame))).copy()
    id_to_path=dict(SAMPLES[dataset])
    for rank,(_,row) in enumerate(chosen.iterrows(),start=1):
        image_id=str(row.image_id)
        x=image_tensor(id_to_path[image_id])
        rgb=display_rgb(x)
        # keep all 20 heatmaps CPU only, one forward/autograd at a time.
        heats={config:{} for config in spec['config_names']}
        probs={config:{} for config in spec['config_names']}
        for seed in SEEDS:
            for config in spec['config_names']:
                model=open_model(dataset,config,seed)
                cam=GradCAM(model)
                try:
                    heat,prob=cam.run(x,idx)
                finally:
                    cam.close()
                release_model(model)
                predicted=float(row[f'prob_{config}_seed{seed}'])
                assert abs(prob-predicted)<0.003, (
                    f'GradCAM mismatch {dataset}/{image_id}/{config}/seed{seed}: {prob} vs {predicted}')
                heats[config][seed]=heat.astype(np.float32)
                probs[config][seed]=prob

        # The mean CAM represents mean of individually normalized maps, NOT mean of logits or attention weights.
        cam_mean={c:np.mean(np.stack([heats[c][s] for s in SEEDS]),axis=0) for c in spec['config_names']}
        cam_std={c:np.std(np.stack([heats[c][s] for s in SEEDS]),axis=0,ddof=1) for c in spec['config_names']}

        fig,axs=plt.subplots(1,5,figsize=(18,4.5),dpi=150,constrained_layout=True)
        axs[0].imshow(rgb);axs[0].set_title('Ảnh Test gốc\n'+spec['label'],fontsize=11)
        for j,(config,title) in enumerate(zip(spec['config_names'],spec['model_titles']),start=1):
            axs[j].imshow(overlay_cam(rgb,cam_mean[config]))
            p=np.array([probs[config][seed] for seed in SEEDS]); tp=int(row[f'tp_seeds_{config}'])
            axs[j].set_title(f'{title} — trung bình 5 seed\np={p.mean():.3f} ± {p.std(ddof=1):.3f} | TP {tp}/5',fontsize=10)
        for ax in axs:ax.axis('off')
        fig.suptitle(f'{dataset} | {spec["label"]} | image_id={image_id} | Mean Grad-CAM across 5 seeds',fontsize=13)
        fig_path=folder/f'GradCAM_{dataset}_case{rank}_mean5.png'
        fig.savefig(fig_path,dpi=200,bbox_inches='tight',facecolor='white')
        plt.show();plt.close(fig)
        print('[MEAN CAM SAVED]',fig_path)

        # one row per seed; compare same image across 4 objectives within each row
        if SAVE_INDIVIDUAL_SEED_FIGURES:
            fig,axs=plt.subplots(len(SEEDS),5,figsize=(14,3.5*len(SEEDS)),dpi=120,constrained_layout=True)
            for i,seed in enumerate(SEEDS):
                axs[i,0].imshow(rgb);axs[i,0].set_title(f'Seed {seed} | ảnh gốc',fontsize=10)
                for j,(config,title) in enumerate(zip(spec['config_names'],spec['model_titles']),start=1):
                    axs[i,j].imshow(overlay_cam(rgb,heats[config][seed]))
                    threshold=THRESHOLDS[(dataset,seed,config)]
                    prob=probs[config][seed]
                    status='TP' if prob>=threshold else 'FN'
                    axs[i,j].set_title(f'{title} | {status}\np={prob:.3f}; threshold={threshold:.3f}',fontsize=10)
                for a in axs[i]:a.axis('off')
            fig.suptitle(f'{dataset} | {spec["label"]} | image_id={image_id} | 5 seeds',fontsize=14)
            detailed=folder/f'GradCAM_{dataset}_case{rank}_all5seeds.png'
            fig.savefig(detailed,dpi=180,bbox_inches='tight',facecolor='white')
            plt.show();plt.close(fig)
            print('[ALL SEEDS CAM SAVED]',detailed)

        # Export numeric map matrices for reproducibility. Avoid 5x4 individual large numpy arrays in Drive.
        np.savez_compressed(folder/f'GradCAM_{dataset}_case{rank}_mean_std_maps.npz',
                            **{f'mean_{config}':cam_mean[config] for config in spec['config_names']},
                            **{f'std_{config}':cam_std[config] for config in spec['config_names']})
        for config in spec['config_names']:
            p=np.array([probs[config][seed] for seed in SEEDS],dtype=float)
            thresholds=np.array([THRESHOLDS[(dataset,seed,config)] for seed in SEEDS])
            all_case_metrics.append(dict(dataset=dataset,image_id=image_id,label=spec['label'],config=config,
                                         positive=1,n_seeds=len(SEEDS),prob_mean=float(p.mean()),
                                         prob_std=float(p.std(ddof=1)),threshold_mean=float(thresholds.mean()),
                                         TP_seeds=int(row[f'tp_seeds_{config}']),FN_seeds=len(SEEDS)-int(row[f'tp_seeds_{config}']),
                                         figure_mean=str(fig_path)))
        all_selected.append(dict(dataset=dataset,image_id=image_id,class_name=spec['label'],case=rank,
                                 bce_FN_seeds=int(row.bce_FN_seeds),improved_TP_total=int(row.improved_TP_total),
                                 figure_mean=str(fig_path)))

all_ap_df=pd.DataFrame(all_ap)
base=all_ap_df[all_ap_df.config=='bce'][['dataset','AP_mean']].rename(columns={'AP_mean':'BCE_AP_mean'})
all_ap_df=all_ap_df.merge(base,on='dataset',validate='many_to_one')
all_ap_df['delta_AP_mean']=all_ap_df.AP_mean-all_ap_df.BCE_AP_mean
all_ap_df.to_csv(OUT_ROOT/'Test_AP_5seeds_mean_std.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(all_case_metrics).to_csv(OUT_ROOT/'selected_cases_probability_5seeds_mean_std.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(all_selected).to_csv(OUT_ROOT/'selected_cases_5seeds.csv',index=False,encoding='utf-8-sig')

# Separate paired delta std from per-seed AP; mean of differences is equal to difference of means.
delta_rows=[]
for dataset,spec in DATASETS.items():
    for config in spec['config_names'][1:]:
        paired=np.array([AP_SCORES[(dataset,seed,config)]-AP_SCORES[(dataset,seed,'bce')] for seed in SEEDS])
        delta_rows.append(dict(dataset=dataset,label=spec['label'],config=config,
                               delta_AP_mean=float(paired.mean()),delta_AP_std=float(paired.std(ddof=1)),
                               positive_delta_seeds=int((paired>0).sum()),n_seeds=5))
pd.DataFrame(delta_rows).to_csv(OUT_ROOT/'Test_paired_delta_AP_5seeds.csv',index=False,encoding='utf-8-sig')

(OUT_ROOT/'README_NOTES.txt').write_text(
    'Stage 1 only. 5 seeds: 7, 42, 132, 2024, 3407.\n'
    'DenseNet201 features.denseblock4, ImageNet normalization, image size 512.\n'
    'Stage1 validation thresholds are read from evaluation CSV rows marked split=test.\n'
    'Mean Grad-CAM: pixelwise mean across 5 individually normalized CAM heatmaps, for qualitative visualization only.\n'
    'CAM colors are NOT quantitatively comparable across models and are NOT AP/accuracy metrics.\n'
    'Prediction mean +/- sample standard deviation across seeds; TP_count based on individual per-seed validation thresholds.\n'
    'AP means and standard deviations are read from full Test evaluation CSV, NOT selected positives.\n'
    'Cases chosen using Test positives for QUALITATIVE illustration; selection must not be used to tune model or threshold.\n',
    encoding='utf-8')
print('[DONE]',OUT_ROOT)
display(all_ap_df)
display(pd.DataFrame(delta_rows))
display(pd.DataFrame(all_case_metrics))


## Kết quả: `outputs/visualizations/stage1-Loss_Functions/Grad_CAM_5seeds/`

Sau khi chạy, mỗi dataset có **2 ca dương tính** được chọn bằng số lần BCE bỏ sót và các loss khác phát hiện đúng. Mỗi ca xuất:

- `*_mean5.png`: ảnh gốc và 4 Grad-CAM **trung bình 5 seed**.
- `*_all5seeds.png`: các Grad-CAM của **5 seed × 4 loss** trên **cùng ảnh**.
- `*_mean_std_maps.npz`: ma trận mean/std CAM, có thể sử dụng để tái dựng hình.
- `positive_test_candidates_5seeds.csv`: xác suất / TP theo seed cho toàn bộ ảnh Test dương tính của bệnh được chọn.

Các bảng tổng hợp ở thư mục gốc gồm `Test_AP_5seeds_mean_std.csv` (AP Test trên **toàn bộ** Test lấy từ CSV Stage 1), `Test_paired_delta_AP_5seeds.csv` (mean ± std độ cải thiện AP ghép cặp seed), `selected_cases_probability_5seeds_mean_std.csv` và `selected_cases_5seeds.csv`.

**Giới hạn:** Trung bình heatmap chỉ minh họa độ nhất quán vùng kích hoạt, không chứng minh tổn thương được định vị chính xác. PCXR có 3 mẫu dương Hyaline membrane disease trên Test, vì vậy kết quả AP và Grad-CAM cần diễn giải thận trọng.
